In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import regex as re # For String searches
import plotly.graph_objects as go
import plotly.express as px
import time
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

data = pd.read_csv('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input/speedtest-data-by-ookla/fixed_year_2021_quarter_03.csv')

Mobile_df = pd.DataFrame([],columns=data.columns)
Broadband_df = pd.DataFrame([],columns=data.columns)

def col_name_corrections(df,correction_pair):
    if set(df.columns).intersection(set(correction_pair.keys())):
        df.rename(columns=correction_pair,inplace=True)
    return df

for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/gksriharsha/eda-speedtests/input'):
    for filename in filenames:
        meta_info = filename.split('/')[-1]
        data = pd.read_csv(dirname+'/'+filename,thousands=r',').convert_dtypes()
        data = col_name_corrections(data,{'Number of Record':'Number of Records'})
        data['Year'] = np.int64(re.search('year_(.*)_quarter',meta_info).group(1))
        data['Quarter'] = np.int64(re.search('quarter_(.*).csv',meta_info).group(1))
        if 'mobile' in meta_info:
            Mobile_df = pd.concat([Mobile_df,data])
        else:
            Broadband_df = pd.concat([Broadband_df,data]) 
print(Broadband_df.shape)

print(Mobile_df.shape)
Mobile_df = Mobile_df.astype({'Year':np.int64,'Quarter':np.int64})
Broadband_df = Broadband_df.astype({'Year':np.int64,'Quarter':np.int64})
Mobile_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)
Broadband_df.sort_values(by=['Year','Quarter'],ascending=[True,True],inplace=True)

factor = 10
Mobile_df = pd.concat([Mobile_df]*factor*2, ignore_index=True)
Mobile_df.info()

Broadband_df = pd.concat([Broadband_df]*factor, ignore_index=True)
Broadband_df.info()

In [ ]:
### cell 1 ###

unique_countries_broadband = Broadband_df.groupby('Name').count()
unique_countries_broadband.head()

In [ ]:
### cell 2 ###

unique_countries_mobile = Mobile_df.groupby('Name').count()
unique_countries_mobile.head()

In [ ]:
### cell 3 ###

# Check for missing values
Mobile_df.isna().any()

In [ ]:
### cell 4 ###

Broadband_df.isna().any()

In [ ]:
### cell 5 ###

Mobile_df.shape

In [ ]:
### cell 6 ###

unique_countries_mobile[unique_countries_mobile.Year < 10]['Year']

In [ ]:
### cell 7 ###

unique_countries_broadband[unique_countries_broadband.Year < 10]['Year']

In [ ]:
### cell 8 ###

Mobile_Stats = Mobile_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
Broadband_Stats = Broadband_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: list(x)[-1] - list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: list(x)[-1] - list(x)[0])
)
# fig = px.histogram(Mobile_Stats['Change_Download'],title='Frequency distribution of Mobile Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()

# fig = px.histogram(Broadband_Stats['Change_Download'],title='Frequency distribution of Broadband Speed change',
#                    labels={'count':'Frequency','value':'$\Delta Speed (Kpbs)$','variable':'property'},
#                    nbins=100)
# fig.show()
Total_Stats = pd.concat([Broadband_Stats['Change_Download'],Mobile_Stats['Change_Download']],axis=1)
Total_Stats.columns=['Mobile','Broadband']

In [ ]:
### cell 9 ###

#ImprovedCountries_M = Mobile_Stats[(Mobile_Stats['Change_Download'] < 3000) &
#                                (Mobile_Stats['Change_Download'] >0)]
#px.bar(ImprovedCountries_M,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 3000) &
                                (Broadband_Stats['Change_Download'] > 0)]

In [ ]:
### cell 10 ###

#ImprovedCountries2 = Mobile_Stats[(Mobile_Stats['Change_Download'] >= 10000)]
#px.bar(ImprovedCountries2,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')
#ImprovedCountries_M = Mobile_Stats[(Mobile_Stats['Change_Download'] < 8000) &
#                                (Mobile_Stats['Change_Download'] >3000)]
#px.bar(ImprovedCountries_M,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Countries that improved download speeds')

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 8000) &
                                (Broadband_Stats['Change_Download'] > 3000)]

In [ ]:
### cell 11 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 16000) &
                                (Broadband_Stats['Change_Download'] > 8000)]

In [ ]:
### cell 12 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 60000) &
                                (Broadband_Stats['Change_Download'] > 16000)]

In [ ]:
### cell 13 ###

ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] >= 60000)]

In [ ]:
### cell 14 ###

#DeterioratedSpeeds = Mobile_Stats[(Mobile_Stats['Change_Download'] < 0 )]
#px.bar(DeterioratedSpeeds,y='Change_Download',labels={'Name':'Country','Change_Download':'Improved Download Speed'},title='Decreasing Countries\' download speeds')
ImprovedCountries_B = Broadband_Stats[(Broadband_Stats['Change_Download'] < 0)]
Countries = ImprovedCountries_B.index

In [ ]:
### cell 15 ###

Mobile_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 16 ###

Broadband_Stats.sort_values(by=['Change_Download'])

In [ ]:
### cell 17 ###

Mobile_Stats_relative = Mobile_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0])
)
Broadband_Stats_relative = Broadband_df.groupby('Name').agg(
    Change_Download=('Avg. Avg D Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Upload=('Avg. Avg U Kbps', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0]),
    Change_Latency=('Avg Lat Ms', lambda x: (list(x)[-1] - list(x)[0])/list(x)[0])
)

In [ ]:
### cell 18 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 2)]

In [ ]:
### cell 19 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 1) & (Broadband_Stats_relative['Change_Download'] < 2)]

In [ ]:
### cell 20 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.5) & (Broadband_Stats_relative['Change_Download'] < 1)]

In [ ]:
### cell 21 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0.2) & (Broadband_Stats_relative['Change_Download'] < 0.5)]

In [ ]:
### cell 22 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] >= 0) & (Broadband_Stats_relative['Change_Download'] < 0.2)]

In [ ]:
### cell 23 ###

ImprovedCountries_B = Broadband_Stats_relative[(Broadband_Stats_relative['Change_Download'] < 0)]

In [ ]:
### cell 24 ###

Broadband_Stats_relative.sort_values(by=['Change_Download'])

In [ ]:
### cell 25 ###

Mobile_Stats_relative.sort_values(by=['Change_Download'])